In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# =========================
# Config
# =========================

catalog_name = "jarvis_training"

silver_schema_name = "silver"
gold_schema_name = "gold"

cards_table = f"{catalog_name}.{silver_schema_name}.clean_dbo_cards_data"
transactions_table = f"{catalog_name}.{silver_schema_name}.clean_dbo_transactions_data"
users_table = f"{catalog_name}.{silver_schema_name}.clean_dbo_users_data"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema_name}")

DataFrame[]

In [0]:
# =========================
# Read Silver Tables
# =========================

cards_df = spark.table(cards_table)
transactions_df = spark.table(transactions_table)
users_df = spark.table(users_table)

In [0]:
# =========================
# Gold Table 1: User Transaction Summary
# =========================

user_transaction_summary_df = (
    transactions_df
    .groupBy("id")
    .agg(
        F.count("*").alias("total_transaction_count"),
        F.sum("amount").alias("total_spending_amount"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.min("amount").alias("min_transaction_amount"),
        F.max("amount").alias("max_transaction_amount"),
        F.countDistinct("card_id").alias("active_card_count"),
        F.min("date").alias("first_transaction_date"),
        F.max("date").alias("latest_transaction_date")
    )
    .join(
        users_df,
        on="id",
        how="left"
    )
    .withColumn("_gold_processed_timestamp", F.current_timestamp())
)

(
    user_transaction_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.user_transaction_summary")
)

In [0]:
# =========================
# Gold Table 2: Card Spending Summary
# =========================

card_spending_summary_df = (
    transactions_df
    .groupBy("id")
    .agg(
        F.count("*").alias("total_transaction_count"),
        F.sum("amount").alias("total_spending_amount"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.min("amount").alias("min_transaction_amount"),
        F.max("amount").alias("max_transaction_amount"),
        F.countDistinct("client_id").alias("distinct_user_count"),
        F.min("date").alias("first_transaction_date"),
        F.max("date").alias("latest_transaction_date")
    )
    .join(
        cards_df,
        on="id",
        how="left"
    )
    .withColumn("_gold_processed_timestamp", F.current_timestamp())
)

(
    card_spending_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.card_spending_summary")
)

In [0]:
# =========================
# Gold Table 3: Monthly Transaction Metrics
# =========================

monthly_transaction_metrics_df = (
    transactions_df
    .withColumn("transaction_month", F.date_trunc("month", F.col("date")))
    .groupBy("transaction_month")
    .agg(
        F.count("*").alias("monthly_transaction_count"),
        F.sum("amount").alias("monthly_total_spending"),
        F.avg("amount").alias("monthly_avg_transaction_amount"),
        F.countDistinct("client_id").alias("monthly_active_users"),
        F.countDistinct("card_id").alias("monthly_active_cards")
    )
    .withColumn("_gold_processed_timestamp", F.current_timestamp())
)

(
    monthly_transaction_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.monthly_transaction_metrics")
)